In [ ]:
!git clone https://github.com/HoaiAn001/Vietnamese-HSD-Augmentation
%cd Vietnamese-HSD-Augmentation
!pip install -q -r requirements.txt
!pip install -q datasets huggingface_hub underthesea

In [ ]:
import random
import pandas as pd
from tqdm.auto import tqdm
from underthesea import word_tokenize
from datasets import load_dataset
from huggingface_hub import login
from google.colab import userdata

In [ ]:
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token, add_to_git_credential=False)

HF_USERNAME = 'HoaiAn001'
REPO_ID     = f'{HF_USERNAME}/tdtu-vietnamese-hsd'

TRAIN_SET = load_dataset(REPO_ID, split='tdtu_train').to_pandas()
DEV_SET   = load_dataset(REPO_ID, split='tdtu_dev').to_pandas()
TEST_SET  = load_dataset(REPO_ID, split='tdtu_test').to_pandas()

In [ ]:
AUGMENT_LABELS   = ['HATE', 'OFFENSIVE']
SAMPLE_PER_LABEL = 500
ALPHA            = 0.1   # Tỉ lệ từ bị thay đổi
NUM_AUG          = 2     # Số bản augment mỗi câu
OPERATIONS       = ['SR', 'RI', 'RS', 'RD']

# TODO: Bạn cần thay bằng file từ điển đồng nghĩa Tiếng Việt thực tế chứa từ lóng/teencode
SYNONYMS = {
    'tốt'   : ['giỏi', 'xuất_sắc', 'ổn'],
    'xấu'   : ['tệ', 'kém', 'dở'],
    'người' : ['con_người', 'cá_nhân'],
    'nói'   : ['bảo', 'phát_biểu', 'kêu'],
    'ghét'  : ['căm', 'thù', 'ghê'],
    'đánh'  : ['đấm', 'đập', 'tấn_công'],
}

def segment(text: str):
    return word_tokenize(text, format='text').split()

def rejoin(tokens: list) -> str:
    return ' '.join(t.replace('_', ' ') for t in tokens)

def sr(tokens, n):
    new = tokens.copy()
    candidates = [t for t in tokens if t in SYNONYMS]
    random.shuffle(candidates)
    for word in candidates[:n]:
        idx = new.index(word)
        new[idx] = random.choice(SYNONYMS[word])
    return new

def ri(tokens, n):
    new = tokens.copy()
    candidates = [t for t in tokens if t in SYNONYMS]
    for _ in range(n):
        if not candidates:
            break
        syn = random.choice(SYNONYMS[random.choice(candidates)])
        new.insert(random.randint(0, len(new)), syn)
    return new

def rs(tokens, n):
    new = tokens.copy()
    for _ in range(n):
        if len(new) >= 2:
            i, j = random.sample(range(len(new)), 2)
            new[i], new[j] = new[j], new[i]
    return new

def rd(tokens, p):
    new = [t for t in tokens if random.random() > p]
    return new if new else [random.choice(tokens)]

def eda(text: str, num_aug: int = NUM_AUG) -> list:
    tokens = segment(text)
    n      = max(1, int(ALPHA * len(tokens)))
    results = []
    for _ in range(num_aug):
        op  = random.choice(OPERATIONS)
        aug = {
            'SR': sr(tokens, n),
            'RI': ri(tokens, n),
            'RS': rs(tokens, n),
            'RD': rd(tokens, ALPHA)
        }[op]
        out = rejoin(aug)
        # Chỉ giữ lại câu nếu nó khác với câu gốc
        if out.strip() != text.strip() and out not in [r[0] for r in results]:
            results.append((out, op))
    return results

In [ ]:
# Đã thêm include_groups=False để chặn lỗi cảnh báo của Pandas
df_minority = (
    TRAIN_SET[TRAIN_SET['label'].isin(AUGMENT_LABELS)]
    .groupby('label', group_keys=False)
    .apply(lambda x: x.sample(min(len(x), SAMPLE_PER_LABEL), random_state=42), include_groups=False)
    .reset_index(drop=True)
)

print(f'Samples to augment: {len(df_minority):,}')
print(df_minority['label'].value_counts())

In [ ]:
augmented_rows = []

for _, row in tqdm(df_minority.iterrows(), total=len(df_minority), desc='Applying EDA'):
    results = eda(row['text'])
    for aug_text, op in results:
        augmented_rows.append({
            'text'         : aug_text,
            'label'        : row['label'],
            'source'       : f"{row['source']}_eda_{op.lower()}",
            'original_text': row['text'],
        })

df_eda = pd.DataFrame(augmented_rows)
print(f'\nTotal Augmented = {len(df_eda):,}')
print("\nLabel Distribution:")
print(df_eda['label'].value_counts())
print("\nOperation Distribution:")
print(df_eda['source'].value_counts())

In [ ]:
# Kiểm tra ngẫu nhiên 5 câu
for _, row in df_eda.sample(5, random_state=42).iterrows():
    # Extract operation name (e.g., 'SR', 'RD') from source
    op_name = row["source"].split("_")[-1].upper()
    print(f'[{row["label"]}] {op_name}')
    print(f'  original : {row["original_text"]}')
    print(f'  augmented: {row["text"]}')
    print("-" * 50)